### Phase 1: Datenbeschaffung & Preprocessing
Zuerst benötigen wir die Rohdaten. Der Zeitraum von mindestens 20-30 Jahren ist ideal (um mehrere Zyklen wie 2000, 2008 und 2020 abzudecken).

*   **Libraries:** `yfinance`, `pandas`, `numpy`
*   **Datenquellen:**
    *   **Risiko-Asset:** 60% S&P 500 (Ticker: `^GSPC`), 40% US-Staatsanleihen (Ticker: `VUSTX`)
    *   **Safe-Haven:** 3-Monats-Treasury Bill Rate als Proxy für Cash-Zinsen(Ticker: `^IRX`)
    *   **Features:** VIX Index (Volatilität), Renditestrukturkurve (10Y-2Y Spread), Momentum-Indikatoren.

In [ ]:
# --- Central Config ---
import sys; sys.path.insert(0, "../config")
from config_loader import cfg

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# 1. Daten laden
# ^GSPC = S&P 500 | VUSTX = Long Bonds | ^VIX = Volatilität | ^IRX = 3-Monats-Zins | ^TNX = 10-Jahres-Zins
tickers = cfg.data.tickers
start_date = cfg.data.start_date
# Enddatum auf HEUTE setzen (yfinance lädt exklusive Enddatum, d.h. bis gestern inklusive)
end_date = cfg.data.end_date
#end_date = "2025-12-31"

# Alles ohne Index-Zugriff runterladen
raw_data = yf.download(tickers, start=start_date, end=end_date)

# --- ROBUSTER MULTI-INDEX FIX (Keyerror-Fix) ---
# Wir prüfen, welcher Preis-Typ verfügbar ist ('Adj Close' bevorzugt, sonst 'Close') - In neueren yfinance-Versionen wurde 'Adj Close' oft durch 'Close' ersetzt, wenn die Daten bereits bereinigt sind. Prüfung beider Case
if 'Adj Close' in raw_data.columns.get_level_values(0):
    data = raw_data['Adj Close'].copy()
    print("Nutze 'Adj Close'")
else:
    data = raw_data['Close'].copy()
    print("Nutze 'Close' (Adj Close nicht gefunden)")
    
# --- Fehlende Werte behandeln ---
# IRX und VIX: Forward-Fill, da Zinsen/Volatilität sich nur an Handelstagen
# aktualisieren und Feiertags-Lücken den letzten bekannten Wert fortschreiben
data['^IRX'] = data['^IRX'].ffill()
data['^VIX'] = data['^VIX'].ffill()
# Verbleibende NaNs am Anfang der Serie entfernen (kein ffill-Anker vorhanden)
data = data.dropna()

# Jetzt haben wir ein "flaches" DataFrame mit den Tickern als Spalten

# 2. Portfolio Erstellung (60% S&P 500, 40% Long Term Bonds)
# --- Log-Renditen (stetige Renditen) ---
# r_t = ln(P_t / P_{t-1}) — additiv, symmetrisch, näher an Normalverteilung
# Log-Renditen nur für Preis-basierte Assets (nicht für Zins-/Volatilitäts-Levels)
price_tickers = ["^GSPC", "VUSTX"]
price_ratio = (data[price_tickers] / data[price_tickers].shift(1)).dropna()
log_returns = np.log(price_ratio)

# Direkter Zugriff auf die Spaltennamen
weights = np.array([cfg.portfolio.weight_equity, cfg.portfolio.weight_bonds])
portfolio_returns = (log_returns[["^GSPC", "VUSTX"]] * weights).sum(axis=1)

df = pd.DataFrame(index=portfolio_returns.index)

# Einzel-Returns von S&P 500 und Bonds
df['Returns_GSPC'] = log_returns['^GSPC']
df['Returns_VUSTX'] = log_returns['VUSTX']

df['Returns'] = portfolio_returns
# Kumulative Rendite: bei Log-Returns über exp(cumsum)
df['Cumulative_Returns'] = np.exp(df['Returns'].cumsum())

# --- CASH-RENDITE INTEGRATION ---
# ^IRX gibt die jährliche Rendite in % an. Umrechnung in tägliche Rendite:
# (Wert / 100) / 252 Handelstage
# Level-Daten: kein Log, direkter Zugriff (Pandas aligned automatisch auf den Index)
df['Cash_Returns'] = np.log(1 + (data['^IRX'] / 100) / 252)
df['VIX'] = data['^VIX']
df['TNX_10Y'] = data['^TNX']
df['IRX_3M'] = data['^IRX']

print(f"Erfolgreich geladen: {df.index[0].date()} bis {df.index[-1].date()}")

print(df)

# Explorative Datenanalyse (EDA)

In [ ]:
import scipy.stats as stats
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf
import matplotlib.pyplot as plt

In [ ]:
# 1. Deskriptive Statistik-Tabelle
print("\n--- DESKRIPTIVE STATISTIK ---")

# Analyse der Basisrenditen
cols_to_analyze = ['Returns_GSPC', 'Returns_VUSTX', 'Returns']

def calculate_descriptive_stats(data_df, columns):
    stats_list = []
    for col in columns:
        series = data_df[col].dropna()
        stats_list.append({
            'Zeitreihe': col,
            'Mittelwert (tägl.)': f"{series.mean():.6f}",
            'Std.Abw. (tägl.)': f"{series.std():.6f}",
            'Min': f"{series.min():.4f}",
            'Max': f"{series.max():.4f}",
            'Schiefe (Skew)': f"{stats.skew(series):.4f}",
            'Kurtosis': f"{stats.kurtosis(series):.4f}"
        })
    return pd.DataFrame(stats_list).set_index('Zeitreihe')

desc_stats_df = calculate_descriptive_stats(df, cols_to_analyze)

# Als Markdown persistieren via config.yaml Pfad
desc_stats_path = cfg.asset_path("eda_descriptive_stats")
desc_stats_df.to_markdown(desc_stats_path)

print(f"Deskriptive Statistik gespeichert unter: {desc_stats_path}")
display(desc_stats_df)

In [ ]:
# 2. Stationaritätstest (Augmented Dickey-Fuller)
print("\n--- STATIONARITÄTSTESTS (ADF) ---")

def run_adf_test(data_df, columns):
    adf_results = []
    for col in columns:
        series = data_df[col].dropna()
        result = adfuller(series)
        adf_results.append({
            'Zeitreihe': col,
            'ADF-Statistik': f"{result[0]:.4f}",
            'p-Wert': f"{result[1]:.4e}",
            'Krit. Wert (5%)': f"{result[4]['5%']:.4f}",
            'Stationär?': 'Ja' if result[1] < 0.05 else 'Nein'
        })
    return pd.DataFrame(adf_results).set_index('Zeitreihe')

adf_table_df = run_adf_test(df, cols_to_analyze)

# Als Markdown persistieren
adf_stats_path = cfg.asset_path("eda_adf_tests")
adf_table_df.to_markdown(adf_stats_path)

print(f"ADF-Test Tabelle gespeichert unter: {adf_stats_path}")
display(adf_table_df)

In [ ]:
# 3. Volatilitätscluster und Autokorrelation (Heteroskedastizität)
fig, axes = plt.subplots(2, 1, figsize=(12, 10))

# Plot 1: Rendite-Verlauf S&P 500
axes[0].plot(df.index, df['Returns_GSPC'], color='blue', alpha=0.6)
axes[0].set_title('S&P 500 Tägliche Renditen (Visualisierung von Volatilitätsclustern)')
axes[0].set_ylabel('Rendite')
axes[0].grid(True, alpha=0.3)

# Plot 2: ACF der quadrierten Renditen
squared_returns = df['Returns_GSPC']**2
plot_acf(squared_returns.dropna(), lags=40, ax=axes[1], alpha=0.05, color='red')
axes[1].set_title('Autokorrelation der quadrierten Renditen (S&P 500) - Nachweis von GARCH-Effekten')

plt.tight_layout()

# Plot persistieren
vol_cluster_path = cfg.asset_path("eda_volatility_clusters")
plt.savefig(vol_cluster_path, dpi=300, bbox_inches='tight')
print(f"Volatilitätscluster-Plot gespeichert unter: {vol_cluster_path}")

plt.show()

In [ ]:
# 4. Historische Drawdowns (SORR Kontext)
# Kumulierte Rendite und Peak berechnen
cum_returns = np.exp(df['Returns'].cumsum())
running_max = cum_returns.cummax()
drawdowns = (cum_returns / running_max) - 1.0

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(drawdowns.index, drawdowns, color='darkred')
ax.fill_between(drawdowns.index, drawdowns, 0, color='red', alpha=0.3)
ax.set_title('Historische Drawdowns des 60/40 Portfolios (Benchmark)')
ax.set_ylabel('Drawdown')
ax.grid(True, alpha=0.3)

# Plot persistieren
drawdown_plot_path = cfg.asset_path("eda_historical_drawdowns")
plt.savefig(drawdown_plot_path, dpi=300, bbox_inches='tight')
print(f"Drawdown-Plot gespeichert unter: {drawdown_plot_path}")

plt.show()

# Die Top 3 Krisen-Tiefpunkte identifizieren
print("\nTop 3 Tiefpunkte des 60/40 Drawdowns:")
print(drawdowns.sort_values().head(3) * 100) # In Prozent anzeigen

In [ ]:
from pathlib import Path

output_path = Path(cfg.data_path("preprocessed"))

# Verzeichnis anlegen, wenn nicht vorhanden
output_path.parent.mkdir(exist_ok=True)

# Speichern als Parquet
df.to_parquet(output_path)
print(f"Dataframe erfolgreich unter {output_path} gespeichert.")